In [0]:
CREATE OR REPLACE TABLE techdados.silver.customers
COMMENT 'Silver – customers base'
AS
SELECT
    customer_id,
    name,
    email,
    CAST(created_at AS DATE) AS created_at,
    ingestion_time,
    source_file
FROM techdados.bronze.customers

In [0]:
CREATE OR REPLACE TABLE techdados.silver.customer_addresses
COMMENT 'Silver – exploded customer addresses'
AS
SELECT
    customer_id,
    addr.col.type   AS address_type,
    addr.col.city   AS city,
    addr.col.state  AS state,
    addr.col.zip    AS zip,
    ingestion_time
FROM techdados.bronze.customers
LATERAL VIEW explode(from_json(addresses, 'array<struct<type:string,city:string,state:string,zip:string>>')) addr

In [0]:
CREATE OR REPLACE TABLE techdados.silver.customer_phones
COMMENT 'Silver – exploded customer phones'
AS
SELECT
    customer_id,
    phone AS phone_number,
    ingestion_time
FROM techdados.bronze.customers,
LATERAL explode(from_json(phones, 'array<string>')) AS phone

In [0]:
CREATE OR REPLACE TABLE techdados.silver.customer_preferences
COMMENT 'Silver – exploded customer preferences'
AS
SELECT
    customer_id,
    pref.newsletter AS newsletter,
    pref.sms        AS sms,
    category        AS category,
    ingestion_time
FROM (
    SELECT 
        customer_id,
        from_json(preferences, 'struct<newsletter:boolean,sms:boolean,categories:array<string>>') AS pref,
        ingestion_time
    FROM techdados.bronze.customers
),
LATERAL explode(pref.categories) AS category

In [0]:
CREATE OR REPLACE TABLE techdados.silver.customer_purchases
COMMENT 'Silver – exploded customer purchase history'
AS
SELECT
    customer_id,
    purchase.col AS purchase_value,
    ingestion_time
FROM techdados.bronze.customers
LATERAL VIEW explode(from_json(purchase_history, 'array<double>')) purchase